In [18]:
from sagemaker.serve.model_builder import ModelBuilder
import boto3
from sagemaker.core import image_uris
from sagemaker.serve.mode.function_pointers import Mode
print("✅ ModelBuilder importado")

✅ ModelBuilder importado


In [19]:
print(list(Mode))

[<Mode.IN_PROCESS: 1>, <Mode.LOCAL_CONTAINER: 2>, <Mode.SAGEMAKER_ENDPOINT: 3>]


In [11]:
#crear rol de ejecutor
execution_role = (
    "arn:aws:iam::066401718601:role/"
    "service-role/AmazonSageMaker-ExecutionRole-20260820T174073"
)


In [14]:
#crear la imagen de xgboost
region = boto3.Session().region_name

xgboost_image = image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    instance_type="ml.m5.large",
    image_scope="inference"
)

print("Region:", region)
print("XGBoost image:")
print(xgboost_image)

[09/14/26 18:38:09] INFO     Ignoring unnecessary instance type: ml.m5.large.                     ]8;id=14033829;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=14033830;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

Region: us-east-2
XGBoost image:
257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.7-1


In [20]:
#Crear el ModelBuilder con la url del modelo
model_s3_uri = (
    "s3://telco-constumer-churn-066401718601-us-east-2-an/"
    "Models/telco-churn-xgboost-20260914175813/output/model.tar.gz"
)

model_builder = ModelBuilder(
    role_arn=execution_role,
    s3_model_data_url=model_s3_uri,
    image_uri=xgboost_image,
    instance_type="ml.m5.large",
    mode=Mode.SAGEMAKER_ENDPOINT,
    content_type="text/csv",
    accept_type="text/csv"
)

print("✅ ModelBuilder configurado para XGBoost")

✅ ModelBuilder configurado para XGBoost


In [21]:
#construir modelo de SageMaker
model = model_builder.build(
    model_name="telco-churn-xgboost-model"
)

print("✅ Modelo SageMaker creado")

[09/14/26 18:40:15] DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=14033842;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=14033843;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1335\1335]8;;\
                             MLflow model input                                                                    

                    INFO     Creating model with name: telco-churn-xgboost-model             ]8;id=14033850;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=14033851;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1922\1922]8;;\

                    DEBUG    No boto3 session provided. Creating a new session.                        ]8;id=14033858;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=14033859;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#357\357]8;;\

                    DEBUG    No config provided. Using default config.                                 ]8;id=14033865;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=14033866;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#365\365]8;;\

[09/14/26 18:40:16] INFO     ✅ Model has been created: 'telco-churn-xgboost-model' using     ]8;id=14033873;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=14033874;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3357\3357]8;;\
                             server None in SAGEMAKER_ENDPOINT mode (ARN:                                          
                             arn:aws:sagemaker:us-east-2:066401718601:model/telco-churn-xgboo                      
                             st-model)                                                                             

✅ Modelo SageMaker creado


In [22]:
endpoint = model_builder.deploy(
    endpoint_name="telco-churn-xgboost-endpoint",
    instance_type="ml.m5.large",
    initial_instance_count=1
)

print("✅ Endpoint desplegado")
print(endpoint)

[09/14/26 18:41:04] INFO     Creating endpoint-config with name telco-churn-xgboost-endpoint ]8;id=14033880;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=14033881;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1093\1093]8;;\

                    INFO     Creating endpoint with name telco-churn-xgboost-endpoint        ]8;id=14033887;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=14033888;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1125\1125]8;;\

[09/14/26 18:41:05] WARNING  Failed to enable live logging: An error occurred                ]8;id=14033894;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=14033895;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2844\2844]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::066401718601:assumed-role/AmazonSageMaker-Executio                       
                             nRole-20260820T174073/SageMaker is not authorized to perform:                         
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-east-2:066401718601:log-group:/aws/sagemaker/En                       
                             dpoints/telco-churn-xgboost-endpoint because no identity-based                        
                             policy allows the logs:FilterLogEvents action. Fallback to                            
                             default logging...                                                                    

Output()

[09/14/26 18:44:35] INFO     ✅ Deployment successful: Endpoint                               ]8;id=14033901;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=14033902;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2687\2687]8;;\
                             'telco-churn-xgboost-endpoint' using None in SAGEMAKER_ENDPOINT                       
                             mode (ARN:                                                                            
                             arn:aws:sagemaker:us-east-2:066401718601:endpoint/telco-churn-xg                      
                             boost-endpoint)                                                                       

✅ Endpoint desplegado
endpoint_name='telco-churn-xgboost-endpoint' endpoint_arn='arn:aws:sagemaker:us-east-2:066401718601:endpoint/telco-churn-xgboost-endpoint' endpoint_config_name='telco-churn-xgboost-endpoint' production_variants=[ProductionVariantSummary(variant_name='AllTraffic', deployed_images=[DeployedImage(specified_image='257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost:1.7-1', resolved_image='257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-xgboost@sha256:b4f13edb198529c460692015797fa1ca6a8ff1ed64a149297174d922121b8fc4', resolution_time=datetime.datetime(2026, 9, 14, 18, 41, 5, 541000, tzinfo=tzlocal()))], current_weight=1.0, desired_weight=1.0, current_instance_count=1, desired_instance_count=1, instance_pools=Unassigned(), variant_status=Unassigned(), current_serverless_config=Unassigned(), desired_serverless_config=Unassigned(), managed_instance_scaling=Unassigned(), routing_config=Unassigned(), capacity_reservation_config=Unassigned())] data_capture_c